In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import k3d
import vtk
import opengate as gate
from scipy.spatial.transform import Rotation

In [2]:
xlsx_path = Path("../../../persistent_data/spreadsheet/MDSL.excel80M10RFR.cut-plate.010.150roi.2.30pin.105ellipse.xlsx")
df = pd.read_excel(xlsx_path,sheet_name="Values") # read the "Values" sheet
df.columns = df.iloc[0]
df = df[1:] # remove the first row which is now the header
df = df.reset_index(drop=True) # reset the index after removing the first row
df = df.apply(pd.to_numeric, errors='coerce') # convert all columns to numeric, coercing errors to NaN
df.columns.name = 'Values Sheet'

# export to csv for easier debugging
csv_path = xlsx_path.with_suffix('.csv')
df.to_csv(csv_path, index=False)

In [3]:
print(df.columns)

Index(['Descriptions', 'Total length', 'First rotation', 'Second rotation',
       'Translation x', 'Translation y', 'Translation z', 'Pixel translation',
       'Collimator translation', 'Collimator length',
       'Collimator plus guide length', 'Guide exit edge length (air)',
       'Guide exit edge length (lead)', 'Guide translation', 'd', 'd + 2t',
       'Ld', 'Ld + 2t', 'Repeat vector', 'Pixel size', 'Crystal thickness',
       'Crystal back face x', 'Crystal back face y', 'Crystal back face z',
       'Crystal centroid x', 'Crystal centroid y', 'Crystal centroid z'],
      dtype='object', name='Values Sheet')


## Get the Rotations and Translations from the Spreadsheet

Also get detector size:

- `Ld+2t`


In [4]:

# Get columns "First rotation" and "Second Rotation", make it a 2d array
rotations_deg = np.array([df["First rotation"].values, df["Second rotation"].values]).T
translations_mm = np.array([df["Translation x"].values, df["Translation y"].values, df["Translation z"].values]).T
detector_size_mm = np.array(df["Ld + 2t"].values).T
print(rotations_deg.shape)
print(translations_mm.shape)
print(detector_size_mm.shape)

(80, 2)
(80, 3)
(80,)


In [5]:
def create_box_mesh(center, size):
    # 1. Define the 8 vertices of a unit cube (from -0.5 to 0.5)
    vertices = np.array(
        [
            [-0.5, -0.5, -0.5],
            [0.5, -0.5, -0.5],
            [0.5, 0.5, -0.5],
            [-0.5, 0.5, -0.5],
            [-0.5, -0.5, 0.5],
            [0.5, -0.5, 0.5],
            [0.5, 0.5, 0.5],
            [-0.5, 0.5, 0.5],
        ],
        dtype=np.float32,
    )

    # 2. Define the 12 triangles (2 per face) that make up the box
    indices = np.array(
        [
            [0, 1, 2],
            [0, 2, 3],  # Bottom
            [4, 5, 6],
            [4, 6, 7],  # Top
            [0, 1, 5],
            [0, 5, 4],  # Side 1
            [1, 2, 6],
            [1, 6, 5],  # Side 2
            [2, 3, 7],
            [2, 7, 6],  # Side 3
            [3, 0, 4],
            [3, 4, 7],  # Side 4
        ],
        dtype=np.uint32,
    )

    # 4. Transform vertices: Scale by size, then Translate by center
    return (vertices * size) + center, indices


# 5. Create Plot and Add Mesh
plot = k3d.plot(name="OpenGATE Geometry Check")


def rotate_vertices(vertices, theta, axis="x"):
    # make sure axis is one of "x", "y", "z"
    assert axis in ["x", "y", "z"], "Axis must be one of 'x', 'y', 'z'"

    c, s = np.cos(theta), np.sin(theta)
    R = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    if axis == "x":
        R = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    elif axis == "y":
        R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    elif axis == "z":
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    # We use the transpose (R.T) or @ operator for matrix multiplication
    return vertices @ R.T


rotations_rad = np.radians(rotations_deg)
for i in range(80):
    center = translations_mm[i]
    size = [60, 200, 60]
    vertices, indices = create_box_mesh(center=[0, 0, 0], size=size)
    vertices = rotate_vertices(
        vertices, rotations_rad[i][1], axis="x"
    )  # rotate around x-axis by the second rotation angle
    vertices = rotate_vertices(
        vertices, rotations_rad[i][0], axis="z"
    )  # rotate around z-axis by the first rotation angle
    vertices += center  # translate to the correct position
    mesh = k3d.mesh(vertices, indices, color=0x00FF00, wireframe=True)
    plot += mesh

plot.display()

Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.


Output()

In [6]:
collimator_tip_outer_size_mm = np.array(df["d + 2t"].values).T
collimator_tip_inner_size_mm = np.array(df["d"].values).T
collimator_body_outer_size_mm = np.array(df["Ld + 2t"].values).T
collimator_body_inner_size_mm = np.array(df["Ld"].values).T
collimator_body_length_mm = np.array(df["Collimator length"].values).T
print(collimator_tip_outer_size_mm.shape)
print(collimator_tip_outer_size_mm[0],collimator_tip_inner_size_mm[0])
print(collimator_body_outer_size_mm[0],collimator_body_inner_size_mm[0])
print(collimator_body_length_mm[0])

(80,)
6.3 2.3
54 50
89.07020616296496


In [11]:

sim=gate.Simulation()
headbox = sim.add_volume(volume="Box", name="HeadBox")
headbox.mother = "world"
if headbox is not None:
    headbox.size = [60, 200, 60]
    headbox.translation = translations_mm[0]
    rz = Rotation.from_euler("x", rotations_deg[0][0], degrees=True)
    rx = Rotation.from_euler("z", rotations_deg[0][1], degrees=True)
    rmat = (rz * rx).as_matrix()
    headbox.rotation = rmat
    collimator_body = sim.add_volume(volume="Trd", name="CollimatorBody")
    collimator_body.mother = "HeadBox"
    collimator_body.dx1 = collimator_body_outer_size_mm[0]
    collimator_body.dy1 = collimator_body_outer_size_mm[0]
    collimator_body.dx2 = collimator_tip_outer_size_mm[0]
    collimator_body.dy2 = collimator_tip_outer_size_mm[0]
    collimator_body.dz = collimator_body_length_mm[0]
    rx_collimator = Rotation.from_euler("x", 90, degrees=True)
    collimator_body.rotation = rx_collimator.as_matrix()
sim.volume_manager.dump_volume_tree()

sim.visu = True
sim.visu_type = "vrml_file_only"
sim.visu_filename = "collimator_geometry.wrl"

In [12]:


# 1. Import the VRML scene using VTK
importer = vtk.vtkVRMLImporter()
importer.SetFileName("collimator_geometry.wrl")
importer.Update()

# 2. Access the renderer to retrieve the geometric shapes (actors)
render_window = importer.GetRenderWindow()
renderer = render_window.GetRenderers().GetFirstRenderer()
actors = renderer.GetActors()

# 3. Initialize an empty K3D plot
plot = k3d.plot()

# 4. Iterate over each piece of the geometry, extract it, and add to K3D
actors.InitTraversal()
for i in range(actors.GetNumberOfItems()):
    actor = actors.GetNextActor()
    polydata = actor.GetMapper().GetInput()
    
    # Push the mesh into K3D using its native vtk_poly_data function
    plot += k3d.vtk_poly_data(polydata, color=0x0072BD, wireframe=True)

# 5. Render the final output
plot.display()

[2026-04-06 09:26:47,496] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-06 09:26:47,499] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-06 09:26:47,502] INFO in helpers: Converting int64 array to int32 for JS compatibility.
[2026-04-06 09:26:47,504] INFO in helpers: Converting int64 array to int32 for JS compatibility.


Output()